Why this notebook exists

My original plan used NYC's Local Law 84 energy data for a daily pipeline, but its metadata says it only changes once a year. A daily Lambda would find nothing on most days. So I'm using 311 heat complaints as the daily data, and LL84 as the building reference data. Before building anything, I want to see what both actually look like.

In [1]:
import os
import requests
import pandas as pd

# token is optional while exploring - the API works without one, just with a lower rate limit.
# once it's a Codespaces secret, os.getenv finds it automatically, no code changes needed
APP_TOKEN = os.getenv("SOCRATA_APP_TOKEN")
HEADERS = {"X-App-Token": APP_TOKEN} if APP_TOKEN else {}

URL_311 = "https://data.cityofnewyork.us/resource/erm2-nwe9.json"    # daily: complaints
URL_LL84 = "https://data.cityofnewyork.us/resource/5zyy-y8am.json"  # yearly: building energy

In [3]:
# only heat complaints - potholes and noise are fun, but not what this question is about
params = {
    "complaint_type": "HEAT/HOT WATER",   # Socrata lets you filter just by naming the column
    "$order": "created_date DESC",
    "$limit": 5,
}
resp = requests.get(URL_311, headers=HEADERS, params=params, timeout=30)
resp.raise_for_status()   # fail loudly on a bad response instead of treating an error page as data
heat = pd.DataFrame(resp.json())
heat.T   # transposed: 5 records become columns, so all ~40 fields fit on screen

,0,1,2,3,4
unique_key,70502629,70507058,70504162,70501110,70505617
created_date,2026-09-22T23:52:58.000,2026-09-22T23:21:24.000,2026-09-22T23:21:19.000,2026-09-22T23:18:55.000,2026-09-22T23:14:39.000
agency,HPD,HPD,HPD,HPD,HPD
agency_name,Department of Housing Preservation and Develop...,Department of Housing Preservation and Develop...,Department of Housing Preservation and Develop...,Department of Housing Preservation and Develop...,Department of Housing Preservation and Develop...
complaint_type,HEAT/HOT WATER,HEAT/HOT WATER,HEAT/HOT WATER,HEAT/HOT WATER,HEAT/HOT WATER
descriptor,APARTMENT ONLY,APARTMENT ONLY,APARTMENT ONLY,ENTIRE BUILDING,APARTMENT ONLY
descriptor_2,NO HOT WATER,NO HOT WATER,NO HOT WATER,NO HOT WATER,NO HOT WATER
location_type,RESIDENTIAL BUILDING,RESIDENTIAL BUILDING,RESIDENTIAL BUILDING,RESIDENTIAL BUILDING,RESIDENTIAL BUILDING
incident_zip,10032,11210,10456,10032,11414
incident_address,69 ST NICHOLAS PLACE,572 EAST 26 STREET,201 MARCY PLACE,860 RIVERSIDE DRIVE,155-47 101 STREET


In [5]:
# 5 rows showed the shape - now I need enough rows to actually trust a percentage.
# last 30 days: big enough to count, small enough to be polite to the API
params = {
    "$select": ":updated_at, unique_key, created_date, descriptor, descriptor_2, status, bbl",
    "$where": "complaint_type = 'HEAT/HOT WATER' AND created_date > '2026-08-25T00:00:00'",
    "$limit": 50000,
}
resp = requests.get(URL_311, headers=HEADERS, params=params, timeout=60)
resp.raise_for_status()
recent = pd.DataFrame(resp.json())
print(f"{len(recent):,} heat/hot water complaints in the last 30 days")

3,759 heat/hot water complaints in the last 30 days


In [7]:
# the hot water question - answered by data instead of by guessing
recent["descriptor_2"].value_counts(dropna=False)

descriptor_2
NO HOT WATER         3632
HEAT ON IN SUMMER     127
Name: count, dtype: int64

In [ ]:

# what share of complaints can actually join to an LL84 building?
recent["bbl"].notna().mean()

np.float64(0.9976057462090981)

In [8]:
# September is the wrong month to test a heating hypothesis - so check last January instead.
# $group makes Socrata count server-side: we get back a few rows, not thousands
params = {
    "$select": "descriptor_2, count(*) AS n",
    "$where": "complaint_type = 'HEAT/HOT WATER' "
              "AND created_date between '2026-01-01T00:00:00' and '2026-01-31T23:59:59'",
    "$group": "descriptor_2",
    "$order": "n DESC",
}
resp = requests.get(URL_311, headers=HEADERS, params=params, timeout=60)
resp.raise_for_status()
pd.DataFrame(resp.json())

,descriptor_2,n
0,NO HEAT,50604
1,NO HEAT AND NO HOT WATER,23444
2,NO HOT WATER,5880


What I found
Join key works: 99.8% of heat complaints have a BBL. The real match to LL84 will be lower, since LL84 only covers large buildings. To be measured next.
Season matters a lot: in September, 97% of complaints are "no hot water" and none are "no heat." In January, 92% involve no heat, at about 20x the daily volume.
Decision: filter to NO HEAT and NO HEAT AND NO HOT WATER. Hot water alone doesn't test my hypothesis about inefficient heating.
Design change: backfill last winter once, then load daily going forward. Otherwise there's nothing to analyze until January.